## Estimators Implementation

(excluding GARCH)

In [1]:
import pandas as pd
import numpy as np

In [2]:
tickers = ['SPY', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']

### Estimators

In [3]:
def close_to_close(week_df):
    T = week_df['dly_ret'].shape[0] 
    if T <= 1:
        return 0.0
    
    r_bar = week_df['dly_ret'].mean() # Average daily return
    vol = np.sqrt((1 / (T - 1)) * ((week_df['dly_ret'] - r_bar) ** 2).sum()) * np.sqrt(252)
    return(vol)

In [4]:
def parkinson(week_df):
    log_hl = np.log(week_df['high'] / week_df['low']) # log(high/low)
    vol = np.sqrt((1 / (4 * np.log(2))) * (log_hl ** 2).sum()) * np.sqrt(252)
    return(vol)

In [5]:
def garman_klass(week_df):
    log_hl = np.log(week_df['high'] / week_df['low']) # log(high/low)
    log_co = np.log(week_df['close'] / week_df['open']) # log(close/open)
    vol = np.sqrt(((0.5 * (log_hl ** 2)) - ((2 * np.log(2) - 1) * (log_co ** 2))).sum()) * np.sqrt(252)
    return(vol)

In [6]:
def rogers_satchell(week_df):
    log_ho = np.log(week_df['high'] / week_df['open']) # log(high/open)
    log_lo = np.log(week_df['low'] / week_df['open']) # log(low/open)
    log_co = np.log(week_df['close'] / week_df['open']) # log(close/open)
    vol = np.sqrt((log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)).sum()) * np.sqrt(252)
    return(vol)

In [7]:
def yang_zhang(week_df):
    T = week_df['dly_ret'].shape[0]
    if T <= 1:
        return 0.0
    c = 0.34 / (1.34 + (T + 1) / (T - 1))
    O = week_df['open']
    H = week_df['high']
    L = week_df['low']
    C = week_df['close']
    
    # Overnight volatility
    o_vol = ((np.log(O / C.shift(1)) - np.log(O / C.shift(1)).mean()) ** 2).sum()
    
    # Open-to-close volatility
    co_vol = ((np.log(C / O) - np.log(C / O).mean()) ** 2).sum()
    
    # Rogers-Satchell volatility
    rs_vol = rogers_satchell(week_df) / np.sqrt(252)
    rs_vol = rs_vol ** 2  # Convert back to variance for sum
    
    vol = np.sqrt((o_vol) + (c * co_vol) + ((1 - c) * rs_vol)) * np.sqrt(252)
    return(vol)

Forecasting volatility using estimators

In [8]:
all_weekly_forecasts = {}
for ticker in tickers:
    df = pd.read_csv(f'../data/individual_ticker_data/{ticker}.csv', parse_dates=True)
    
    # Just placeholder to start the dataframe
    df_start = df.groupby('week_num').apply(
        close_to_close
    ).to_frame('placeholder')
    
    df_forecasts = pd.DataFrame(index=df_start.index) # Use the same week_num index

    df_forecasts['close_to_close'] = df.groupby('week_num').apply(close_to_close)
    df_forecasts['parkinson'] = df.groupby('week_num').apply(parkinson)
    df_forecasts['garman_klass'] = df.groupby('week_num').apply(garman_klass)
    df_forecasts['rogers_satchell'] = df.groupby('week_num').apply(rogers_satchell)
    df_forecasts['yang_zhang'] = df.groupby('week_num').apply(yang_zhang)

    
    week_end_dates = df.groupby('week_num')['trade_date'].max()
    
    df_start = df_start.set_index(week_end_dates)
    df_forecasts = df_forecasts.set_index(week_end_dates)
    
    final_df = pd.merge(df_forecasts, df_start, left_index=True, right_index=True)
    final_df.dropna(inplace=True)
    final_df.drop(columns=['placeholder'], inplace=True)
    
    final_df.to_csv(f'../data/basic_forecasts/{ticker}_basic_forecast.csv')
    all_weekly_forecasts[ticker] = final_df


all_weekly_forecasts['SPY']

C:\Users\rahul\AppData\Local\Temp\ipykernel_57308\3666414033.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_start = df.groupby('week_num').apply(
C:\Users\rahul\AppData\Local\Temp\ipykernel_57308\3666414033.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_forecasts['close_to_close'] = df.groupby('week_num').apply(close_to_close)
C:\Users\rahul\AppData\Local\Temp\ipykernel_57308\3666414033.py:

,close_to_close,parkinson,garman_klass,rogers_satchell,yang_zhang
trade_date,,,,,
2005-01-07,0.102138,0.269486,0.257288,0.246317,0.245690
2005-01-14,0.104154,0.191756,0.196284,0.200665,0.210659
2005-01-21,0.146252,0.215565,0.179217,0.151105,0.180176
2005-01-28,0.033285,0.147761,0.166406,0.184227,0.183102
2005-02-04,0.078090,0.163493,0.151878,0.142885,0.146854
...,...,...,...,...,...
2025-09-05,0.115752,0.184483,0.189185,0.195990,0.195557
2025-09-12,0.050211,0.115679,0.122305,0.121710,0.135795
2025-09-19,0.050261,0.146881,0.168718,0.178095,0.176540
